# Reto de Hibridación de Arquitecturas CNN (Chimera)

## Contexto

Hasta ahora hemos aprendido las bases del boque extractor de caracteristicas, y vimos los pricipios de funcionameinto de algunas arquitecturas CNN modernas por separado: VGG, ResNet,Inception y DenseNet. Cada una resuelve un problema distinto de diseño (profundidad, gradientes que se desvanecen, costo computacional, reutilización de características).
Con este taller cerramos la sección de clasificación con un reto sustancioso: **construir su propia arquitectura combinando bloques de esas familias**, bajo un presupuesto de
puntos, y comparar su desempeño contra transfer learning y fine-tuning sobre el
mismo dataset real.

## Objetivos de aprendizaje

Al terminar este taller, cada grupo debe poder:

1. Justificar por qué eligieron ciertos bloques y no otros, en términos de costo  computacional real (no solo intuición).
2. Ensamblar una arquitectura híbrida funcional respetando un presupuesto fijo.
3. Entrenar esa arquitectura desde cero y medir su desempeño real en un dataset
   real, no sintético.
4. Aplicar transfer learning y fine-tuning de la arquitectura sobre otro dataset de imagenes.

## Formato y tiempo

- Trabajo en **grupos** .
- **1 hora** de tiempo base. Si el avance del grupo lo justifica, se otorga
  **tiempo adicional**.
- **Máximo 25 épocas de entrenamiento** para cada uno de los tres componentes (arquitectura propia, transfer learning, fine-tuning). El código de este notebook hace cumplir este límite automáticamente.
- Dataset real: Intel Image Classification (6 clases de paisajes naturales y urbanos).

## Qué se entrega

| Componente | Umbral de F1 macro | Cómo se califica |
|---|---|---|
| Arquitectura propia (desde cero) | 95% es la meta, no un mínimo obligatorio | Escala gradual: la nota crece proporcionalmente si logra menos una metrica F1 ≥ 80% nota=3, si F1 ≥ 85% nota = 4 y si F1 ≥ 95% sobre 5|
| Transfer learning | F1 ≥ 85% | Escala gradual: la nota crece proporcionalmente si logra menos una metrica F1 ≥ 77% nota=3, si F1 ≥ 82% nota = 4 y si F1 ≥ 86% sobre 5 |
| Fine-tuning | F1 ≥ 87% | Escala gradual: la nota crece proporcionalmente si logra menos una metrica F1 ≥ 78% nota=3, si F1 ≥ 84% nota = 4 y si F1 ≥ 88% sobre 5 |


### Nota importante sobre transfer learning y fine-tuning en este reto

A diferencia de un transfer learning clásico (que parte de un backbone preentrenado en ImageNet, como ResNet18), **aquí transfer learning y
fine-tuning se hacen sobre su propia arquitectura Chimera**,  la misma que entrenaron desde cero en Intel Image Classification. La idea es medir qué
tan transferibles son las características que su propia red aprendió, no las de un modelo ajeno.

El dataset nuevo para esas dos etapas es **PathMNIST** (imágenes reales de histología de tejido colorrectal, 9 clases), a una resolución de **256×256**. Nota técnica: MedMNIST solo ofrece nativamente hasta 224×224, se toma esa versión y se hace *upsampling* a 256×256 con `transforms.Resize`, lo cual queda explícito en el código para que no parezca una resolución nativa que no existe.


## Presupuesto de puntos (recalibrado: rango 150-200)

Los costos de los bloques se derivaron midiendo, y verificando dos veces
con fórmulas independientes, los parámetros reales de cada bloque en
PyTorch a un ancho de canal constante (64). El catálogo ahora tiene **12
bloques** (6 familias × 2 variantes: *entrada*, más simple, e *intermedia*,
más profunda o con menos reducción):

| Bloque | Puntos | Parámetros reales |
|---|---|---|
| MobileNet entrada (V1, depthwise separable) | 8 | 5,056 |
| Inception entrada (bottleneck agresivo) | 10 | 9,352 |
| ConvNeXt entrada (expansión x2) | 15 | 19,968 |
| Inception intermedia (bottleneck suave) | 20 | 29,712 |
| ConvNeXt intermedia (expansión x4, como el paper) | 25 | 36,480 |
| DenseNet entrada (1 capa densa, growth=32) | 28 | 45,600 |
| MobileNet intermedia (V2, residual invertido x6) | 32 | 55,104 |
| VGG entrada (2 conv 3×3) | 38 | 74,112 |
| ResNet entrada (BasicBlock, 2 conv 3×3) | 38 | 74,112 |
| DenseNet intermedia (2 capas densas apiladas) | 45 | 95,360 |
| VGG intermedia (3 conv 3×3) | 50 | 111,168 |
| ResNet intermedia (BasicBlock de 3 conv) | 50 | 111,168 |

Otra vez: VGG y ResNet cuestan exactamente lo mismo en cada variante,la
conexión de salto sigue sin costar ni un parámetro.

**Componentes adicionales** (no son "bloques de familia", son decisiones de
diseño libres):

| Componente | Puntos |
|---|---|
| Bloque convolucional custom (conv+BN+ReLU, kernel a elección) | 5 |
| MaxPooling o AveragePooling extra | 2 |
| Global Average Pooling (en vez de aplanar todo el mapa) | 3 |
| Dropout | 2 |
| Dropout espacial (Dropout2d) | 3 |

Optimizador, scheduler y función de pérdida mantienen el mismo costeo de
antes (SGD=1, RMSprop=5, Adam=4, AdamW=6 · Ninguno=0, StepLR=3, Cosine=5,
Plateau=6 · CrossEntropy=1, Label Smoothing=4, Focal=8).

**Regla puntual y estricta: el presupuesto total debe quedar entre 150 y 200 puntos. Si se pasan de 200 o no llegan a 150, la celda de verificación lanza un
error y esa arquitectura no se califica**, no hay excepciones ni redondeos. Ojo tambien si se entrega por fuera de horario establecio se asignara un 0.


## 1. Configuración inicial

In [ ]:
import os
import sys
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, classification_report

# chimera_blocks.py debe estar en la misma carpeta que este notebook. Se
# factorizaron ahi los bloques, la arquitectura y el presupuesto, no se deben
# duplican aqui

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from chimera_blocks import (
    BLOCK_COSTS, COMPONENT_COSTS, OPTIMIZER_COSTS, SCHEDULER_COSTS, LOSS_COSTS,
    PRESUPUESTO_MIN, PRESUPUESTO_MAX, MAX_EPOCHS,
    calcular_presupuesto, ChimeraNet, CRITERIONS, OPTIMIZERS, construir_scheduler,
    evaluar_f1, entrenar_modelo, nota_graduada, F1_MINIMO_ACEPTABLE, F1_META,
    guardar_checkpoint_firmado,
)

CODIGO_GRUPO = 77

def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fijar_semilla(CODIGO_GRUPO)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0
PIN_MEMORY = False
print(f"Dispositivo: {device}")
print(f"Presupuesto permitido: {PRESUPUESTO_MIN}-{PRESUPUESTO_MAX} pts")
print(f"Limite de epocas: {MAX_EPOCHS}")


Dispositivo: cuda
Presupuesto permitido: 150-200 pts
Limite de epocas: 25


In [2]:
# Vista rapida de las tablas de costos completas (ya definidas en chimera_blocks.py)
print("BLOQUES:")
for nombre, pts in sorted(BLOCK_COSTS.items(), key=lambda kv: kv[1]):
    print(f"  {nombre:<24}{pts:>4} pts")
print("\nCOMPONENTES:")
for nombre, pts in COMPONENT_COSTS.items():
    print(f"  {nombre:<24}{pts:>4} pts")
print("\nOPTIMIZADOR / SCHEDULER / LOSS:")
print(" ", OPTIMIZER_COSTS)
print(" ", SCHEDULER_COSTS)
print(" ", LOSS_COSTS)


BLOQUES:
  mobilenet_entrada          8 pts
  inception_entrada         10 pts
  convnext_entrada          15 pts
  inception_intermedia      20 pts
  convnext_intermedia       25 pts
  densenet_entrada          28 pts
  mobilenet_intermedia      32 pts
  vgg_entrada               38 pts
  resnet_entrada            38 pts
  densenet_intermedia       45 pts
  vgg_intermedia            50 pts
  resnet_intermedia         50 pts

COMPONENTES:
  conv_custom                5 pts
  maxpool_extra              2 pts
  avgpool_extra              2 pts
  global_avg_pool            3 pts
  dropout                    2 pts
  dropout_espacial           3 pts

OPTIMIZADOR / SCHEDULER / LOSS:
  {'sgd': 1, 'rmsprop': 5, 'adam': 4, 'adamw': 6}
  {'none': 0, 'steplr': 3, 'cosine': 5, 'plateau': 6}
  {'crossentropy': 1, 'label_smoothing': 4, 'focal': 8}


## 2. Dos ejemplos de referencia

Antes de armar la suya, miren estos dos ejemplos completos: cómo se define la configuración, cómo se valida el presupuesto, y cómo se instancia
`ChimeraNet`. **No usen estas combinaciones tal cual** para su entrega, son solo para que vean el patrón.


In [3]:
# --- Ejemplo A: pocos bloques, cada uno de una familia distinta ---
ejemplo_a_bloques = ["vgg_entrada", "mobilenet_intermedia", "convnext_entrada", "resnet_entrada"]
ejemplo_a_optimizador = "adamw"
ejemplo_a_scheduler = "plateau"
ejemplo_a_loss = "focal"
ejemplo_a_conv_custom = True
ejemplo_a_pooling_extra = None
ejemplo_a_global_avg_pool = True
ejemplo_a_dropout = ("dropout", 0.3)

print("=== Ejemplo A ===")
total_a, valido_a, _ = calcular_presupuesto(
    ejemplo_a_bloques, ejemplo_a_optimizador, ejemplo_a_scheduler, ejemplo_a_loss,
    usar_conv_custom=ejemplo_a_conv_custom, pooling_extra=ejemplo_a_pooling_extra,
    usar_global_avg_pool=ejemplo_a_global_avg_pool, dropout=ejemplo_a_dropout,
)

modelo_ejemplo_a = ChimeraNet(
    ejemplo_a_bloques, num_classes=6,
    usar_conv_custom=ejemplo_a_conv_custom, pooling_extra=ejemplo_a_pooling_extra,
    usar_global_avg_pool=ejemplo_a_global_avg_pool, dropout=ejemplo_a_dropout,
)
n_params_a = sum(p.numel() for p in modelo_ejemplo_a.parameters() if p.requires_grad)
print(f"Parametros totales del Ejemplo A: {n_params_a:,}")


=== Ejemplo A ===
Bloques:      ['vgg_entrada', 'mobilenet_intermedia', 'convnext_entrada', 'resnet_entrada'] -> 123 pts
Optimizador:  adamw -> 6 pts
Scheduler:    plateau -> 6 pts
Loss:         focal -> 8 pts
Componentes:  conv_custom=True, pooling_extra=None, global_avg_pool=True, dropout=('dropout', 0.3) -> 10 pts
TOTAL: 153 pts (rango permitido: 150-200)
VALIDO
Parametros totales del Ejemplo A: 262,662


In [4]:
# --- Ejemplo B: pocos bloques, pero mas pesados (variantes intermedia) ---
ejemplo_b_bloques = ["vgg_intermedia", "resnet_intermedia", "densenet_intermedia"]
ejemplo_b_optimizador = "adam"
ejemplo_b_scheduler = "cosine"
ejemplo_b_loss = "crossentropy"
ejemplo_b_conv_custom = False
ejemplo_b_pooling_extra = None
ejemplo_b_global_avg_pool = True
ejemplo_b_dropout = ("dropout_espacial", 0.2)

print("=== Ejemplo B ===")
total_b, valido_b, _ = calcular_presupuesto(
    ejemplo_b_bloques, ejemplo_b_optimizador, ejemplo_b_scheduler, ejemplo_b_loss,
    usar_conv_custom=ejemplo_b_conv_custom, pooling_extra=ejemplo_b_pooling_extra,
    usar_global_avg_pool=ejemplo_b_global_avg_pool, dropout=ejemplo_b_dropout,
)

modelo_ejemplo_b = ChimeraNet(
    ejemplo_b_bloques, num_classes=6,
    usar_conv_custom=ejemplo_b_conv_custom, pooling_extra=ejemplo_b_pooling_extra,
    usar_global_avg_pool=ejemplo_b_global_avg_pool, dropout=ejemplo_b_dropout,
)
n_params_b = sum(p.numel() for p in modelo_ejemplo_b.parameters() if p.requires_grad)
print(f"Parametros totales del Ejemplo B: {n_params_b:,}")
print()
print("Noten la diferencia de estrategia: el Ejemplo A usa 4 bloques livianos")
print("de familias distintas + un conv custom; el Ejemplo B usa solo 3 bloques")
print("pesados (variantes 'intermedia'). Ambos caen en el rango 150-200 por")
print("caminos distintos -- esa es exactamente la decision de diseno que")
print("tienen que tomar con su propia arquitectura.")


=== Ejemplo B ===
Bloques:      ['vgg_intermedia', 'resnet_intermedia', 'densenet_intermedia'] -> 145 pts
Optimizador:  adam -> 4 pts
Scheduler:    cosine -> 5 pts
Loss:         crossentropy -> 1 pts
Componentes:  conv_custom=False, pooling_extra=None, global_avg_pool=True, dropout=('dropout_espacial', 0.2) -> 6 pts
TOTAL: 161 pts (rango permitido: 150-200)
VALIDO
Parametros totales del Ejemplo B: 328,390

Noten la diferencia de estrategia: el Ejemplo A usa 4 bloques livianos
de familias distintas + un conv custom; el Ejemplo B usa solo 3 bloques
pesados (variantes 'intermedia'). Ambos caen en el rango 150-200 por
caminos distintos -- esa es exactamente la decision de diseno que
tienen que tomar con su propia arquitectura.


## 3. Su arquitectura

Definan aquí su propia combinación. Deben quedar entre 150 y 200 puntos, la celda de verificación lanza un `AssertionError` si no cumplen, y no van
a poder seguir hasta corregirlo.


In [ ]:
# Configuracion del grupo -- candidato "mixto_barato_ancho" (196 pts)
MI_BLOQUES = ["mobilenet_entrada", "inception_entrada", "inception_intermedia", "convnext_intermedia", "mobilenet_intermedia", "resnet_entrada", "vgg_entrada"]
MI_OPTIMIZADOR = "adamw"
MI_SCHEDULER = "cosine"
MI_LOSS = "label_smoothing"
MI_CONV_CUSTOM = True
MI_POOLING_EXTRA = None     # None | "maxpool" | "avgpool"
MI_GLOBAL_AVG_POOL = True
MI_DROPOUT = ("dropout", 0.3)                # None | ("dropout", p) | ("dropout_espacial", p)

total_pts, presupuesto_valido, _ = calcular_presupuesto(
    MI_BLOQUES, MI_OPTIMIZADOR, MI_SCHEDULER, MI_LOSS,
    usar_conv_custom=MI_CONV_CUSTOM, pooling_extra=MI_POOLING_EXTRA,
    usar_global_avg_pool=MI_GLOBAL_AVG_POOL, dropout=MI_DROPOUT,
)
assert presupuesto_valido, (
    f"Su combinacion no cumple el presupuesto ({PRESUPUESTO_MIN}-{PRESUPUESTO_MAX} pts). "
    f"Total actual: {total_pts}. Ajustenla antes de seguir -- esto NO se puede entregar asi."
)


Bloques:      ['vgg_intermedia', 'resnet_intermedia', 'densenet_intermedia'] -> 145 pts
Optimizador:  adam -> 4 pts
Scheduler:    cosine -> 5 pts
Loss:         label_smoothing -> 4 pts
Componentes:  conv_custom=False, pooling_extra=None, global_avg_pool=True, dropout=None -> 3 pts
TOTAL: 161 pts (rango permitido: 150-200)
VALIDO


## 4. Datos: Intel Image Classification (arquitectura propia)


ya enviados por link


In [6]:
DATA_DIR_INTEL = "./data/intel_subset"
IMG_SIZE_INTEL = 64
BATCH_SIZE = 32

if not os.path.isdir(DATA_DIR_INTEL):
    raise FileNotFoundError(
        f"No se encontro '{DATA_DIR_INTEL}'. El profesor debe correr prepare_intel_dataset.py antes de la sesion."
    )

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE_INTEL, IMG_SIZE_INTEL)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE_INTEL, IMG_SIZE_INTEL)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_full = datasets.ImageFolder(os.path.join(DATA_DIR_INTEL, "train"), transform=transform_train)
train_full_eval = datasets.ImageFolder(os.path.join(DATA_DIR_INTEL, "train"), transform=transform_eval)
test_dataset_intel = datasets.ImageFolder(os.path.join(DATA_DIR_INTEL, "test"), transform=transform_eval)

CLASSES_INTEL = train_full.classes
NUM_CLASSES_INTEL = len(CLASSES_INTEL)
print(f"Clases Intel ({NUM_CLASSES_INTEL}): {CLASSES_INTEL}")

generador_split = torch.Generator().manual_seed(CODIGO_GRUPO)
n_val = int(0.15 * len(train_full))
n_train = len(train_full) - n_val
indices = torch.randperm(len(train_full), generator=generador_split).tolist()
idx_train, idx_val = indices[:n_train], indices[n_train:]

train_subset = torch.utils.data.Subset(train_full, idx_train)
val_subset = torch.utils.data.Subset(train_full_eval, idx_val)
print(f"  -> train: {len(train_subset)}  |  val: {len(val_subset)}  |  test (held-out): {len(test_dataset_intel)}")

train_loader_intel = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader_intel = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader_intel = DataLoader(test_dataset_intel, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# Batch fijo para el fingerprint anti-fraude: siempre las mismas primeras
# imagenes del test set, en el mismo orden (shuffle=False), para que el hash
# sea reproducible por el profesor al recalcularlo.
BATCH_FIJO_INTEL, _ = next(iter(test_loader_intel))


Clases Intel (6): ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
  -> train: 11929  |  val: 2105  |  test (held-out): 3000


## 5. Construcción y entrenamiento: arquitectura propia

In [7]:
modelo_propio = ChimeraNet(
    MI_BLOQUES, num_classes=NUM_CLASSES_INTEL,
    usar_conv_custom=MI_CONV_CUSTOM, pooling_extra=MI_POOLING_EXTRA,
    usar_global_avg_pool=MI_GLOBAL_AVG_POOL, dropout=MI_DROPOUT,
).to(device)

# Forward de verificacion ANTES de crear el optimizador: si MI_GLOBAL_AVG_POOL
# es False, la capa final es LazyLinear y necesita este forward para
# materializar sus parametros; si el optimizador se crea antes, no vera esos
# pesos y el entrenamiento fallara silenciosamente en ese componente.
with torch.no_grad():
    x_batch, _ = next(iter(train_loader_intel))
    salida = modelo_propio(x_batch.to(device))
    assert salida.shape == (x_batch.shape[0], NUM_CLASSES_INTEL), f"Forma inesperada: {salida.shape}"
    print(f"Verificacion de forma OK: {tuple(x_batch.shape)} -> {tuple(salida.shape)}")

n_params_totales = sum(p.numel() for p in modelo_propio.parameters() if p.requires_grad)
print(f"Arquitectura propia: {n_params_totales:,} parametros entrenables")

optimizer_propio = OPTIMIZERS[MI_OPTIMIZADOR](modelo_propio.parameters())
scheduler_propio = construir_scheduler(MI_SCHEDULER, optimizer_propio, max_epochs=MAX_EPOCHS)
criterion_propio = CRITERIONS[MI_LOSS]()

historial_propio, tiempo_inicio_propio = entrenar_modelo(
    modelo_propio, train_loader_intel, val_loader_intel, optimizer_propio, criterion_propio,
    device, max_epochs=MAX_EPOCHS, scheduler=scheduler_propio, nombre="arquitectura_propia"
)


Verificacion de forma OK: (32, 3, 64, 64) -> (32, 6)
Arquitectura propia: 328,390 parametros entrenables
[arquitectura_propia] epoca  1/25  loss_train=1.2183  f1_val=0.6642
[arquitectura_propia] epoca  2/25  loss_train=1.0004  f1_val=0.5778
[arquitectura_propia] epoca  3/25  loss_train=0.9185  f1_val=0.7412
[arquitectura_propia] epoca  4/25  loss_train=0.8633  f1_val=0.7908
[arquitectura_propia] epoca  5/25  loss_train=0.8278  f1_val=0.8149
[arquitectura_propia] epoca  6/25  loss_train=0.8046  f1_val=0.8562
[arquitectura_propia] epoca  7/25  loss_train=0.7778  f1_val=0.8251
[arquitectura_propia] epoca  8/25  loss_train=0.7670  f1_val=0.8589
[arquitectura_propia] epoca  9/25  loss_train=0.7464  f1_val=0.8209
[arquitectura_propia] epoca 10/25  loss_train=0.7385  f1_val=0.8503
[arquitectura_propia] epoca 11/25  loss_train=0.7204  f1_val=0.8465
[arquitectura_propia] epoca 12/25  loss_train=0.7098  f1_val=0.8823
[arquitectura_propia] epoca 13/25  loss_train=0.7006  f1_val=0.8723
[arquitectu

In [8]:
f1_propio, y_true_propio, y_pred_propio = evaluar_f1(modelo_propio, test_loader_intel, device)
print(f"F1 macro final (arquitectura propia): {f1_propio:.4f}")
print(classification_report(y_true_propio, y_pred_propio, target_names=CLASSES_INTEL))
print(f"Nota graduada del componente 'arquitectura propia': {nota_graduada(f1_propio):.2f} (sobre 1.0)")

os.makedirs("./entregas", exist_ok=True)
checkpoint_propio = guardar_checkpoint_firmado(
    modelo_propio, f"./entregas/grupo{CODIGO_GRUPO}_arquitectura_propia.pth",
    historial_propio, epocas_entrenadas=MAX_EPOCHS, codigo_grupo=CODIGO_GRUPO,
    tiempo_inicio=tiempo_inicio_propio, batch_fijo=BATCH_FIJO_INTEL, device=device,
    componente="arquitectura_propia",
)


F1 macro final (arquitectura propia): 0.9005
              precision    recall  f1-score   support

   buildings       0.89      0.88      0.89       437
      forest       0.97      0.99      0.98       474
     glacier       0.89      0.83      0.86       553
    mountain       0.84      0.88      0.86       525
         sea       0.91      0.93      0.92       510
      street       0.91      0.89      0.90       501

    accuracy                           0.90      3000
   macro avg       0.90      0.90      0.90      3000
weighted avg       0.90      0.90      0.90      3000

Nota graduada del componente 'arquitectura propia': 1.00 (sobre 1.0)
Checkpoint guardado en ./entregas/grupo0_arquitectura_propia.pth
  epocas_entrenadas=25  duracion=1176.1s  fingerprint=2207637a1fe13af4...


## 6. Datos: PathMNIST (transfer learning y fine-tuning)

PathMNIST: imágenes reales de histología de tejido colorrectal, 9 clases (adiposo, fondo, debris, linfocitos, mucosa, músculo liso, mucosa colónica normal, estroma, adenocarcinoma). Estan preparadas de antemano igual que Intel Image Classification (`prepare_pathmnist_dataset.py`), a 256×256 (upsampled desde el máximo nativo de MedMNIST, que es 224×224).

```
./data/pathmnist_subset/train/{adipose,background,debris,lymphocytes,mucus,
                                 smooth_muscle,normal_colon_mucosa,
                                 cancer_stroma,adenocarcinoma}/
./data/pathmnist_subset/test/{...mismas 9 carpetas...}
```


In [9]:
DATA_DIR_PATH = "./data/pathmnist_subset"
IMG_SIZE_PATH = 256

if not os.path.isdir(DATA_DIR_PATH):
    raise FileNotFoundError(
        f"No se encontro '{DATA_DIR_PATH}'. El profesor debe correr prepare_pathmnist_dataset.py antes de la sesion."
    )

transform_train_path = transforms.Compose([
    transforms.Resize((IMG_SIZE_PATH, IMG_SIZE_PATH)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
transform_eval_path = transforms.Compose([
    transforms.Resize((IMG_SIZE_PATH, IMG_SIZE_PATH)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_full_path = datasets.ImageFolder(os.path.join(DATA_DIR_PATH, "train"), transform=transform_train_path)
train_full_path_eval = datasets.ImageFolder(os.path.join(DATA_DIR_PATH, "train"), transform=transform_eval_path)
test_dataset_path = datasets.ImageFolder(os.path.join(DATA_DIR_PATH, "test"), transform=transform_eval_path)

CLASSES_PATH = train_full_path.classes
NUM_CLASSES_PATH = len(CLASSES_PATH)
print(f"Clases PathMNIST ({NUM_CLASSES_PATH}): {CLASSES_PATH}")

generador_split_path = torch.Generator().manual_seed(CODIGO_GRUPO + 1)
n_val_p = int(0.15 * len(train_full_path))
n_train_p = len(train_full_path) - n_val_p
indices_p = torch.randperm(len(train_full_path), generator=generador_split_path).tolist()
idx_train_p, idx_val_p = indices_p[:n_train_p], indices_p[n_train_p:]

train_subset_path = torch.utils.data.Subset(train_full_path, idx_train_p)
val_subset_path = torch.utils.data.Subset(train_full_path_eval, idx_val_p)

# Batch size mas chico para PathMNIST: 256x256 pesa mucho mas que los 64x64
# de Intel (16x mas pixeles por imagen), y en laptops de 4GB VRAM un batch de
# 32 a esta resolucion puede no caber en memoria.
BATCH_SIZE_PATH = 16

train_loader_path = DataLoader(train_subset_path, batch_size=BATCH_SIZE_PATH, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader_path = DataLoader(val_subset_path, batch_size=BATCH_SIZE_PATH, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader_path = DataLoader(test_dataset_path, batch_size=BATCH_SIZE_PATH, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

BATCH_FIJO_PATH, _ = next(iter(test_loader_path))

print(f"  -> train: {len(train_subset_path)}  |  val: {len(val_subset_path)}  |  test (held-out): {len(test_dataset_path)}")
print()
print("ADVERTENCIA DE TIEMPO: 256x256 es 16 veces mas pixeles por imagen que")
print("los 64x64 usados en Intel. Si el entrenamiento de transfer o fine-tuning")
print("se ve demasiado lento, avisen al profesor -- puede que haya que reducir")
print("el batch size aun mas, o el numero de imagenes por clase en el dataset.")


Clases PathMNIST (9): ['adenocarcinoma', 'adipose', 'background', 'cancer_stroma', 'debris', 'lymphocytes', 'mucus', 'normal_colon_mucosa', 'smooth_muscle']
  -> train: 15300  |  val: 2700  |  test (held-out): 4260

ADVERTENCIA DE TIEMPO: 256x256 es 16 veces mas pixeles por imagen que
los 64x64 usados en Intel. Si el entrenamiento de transfer o fine-tuning
se ve demasiado lento, avisen al profesor -- puede que haya que reducir
el batch size aun mas, o el numero de imagenes por clase en el dataset.


## 7. Transfer learning (sobre su propia arquitectura, dataset nuevo)

Congelamos `stem` + `features` de `modelo_propio` (ya entrenado en Intel) y le ponemos una cabeza nueva para las 9 clases de PathMNIST. Mismo patrón de
BatchNorm de siempre: el modelo completo en `eval()`, solo la cabeza nueva en `train()`,  si no, las estadísticas de BatchNorm del backbone se corrompen con las estadísticas del batch pequeño del dataset nuevo.


In [10]:
for param in modelo_propio.stem.parameters():
    param.requires_grad = False
for param in modelo_propio.features.parameters():
    param.requires_grad = False

BASE_CHANNELS = modelo_propio.base_channels
modelo_propio.head = nn.Sequential(
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(BASE_CHANNELS, NUM_CLASSES_PATH)
).to(device)
# Importante: config_dict() (usado para reconstruir el modelo en la
# calificacion offline) lee estos atributos para saber como construir la
# cabeza. Como acabamos de reemplazar 'head' a mano por una version simple
# (sin dropout, con global average pooling), hay que actualizar los atributos
# para que coincidan -- si no, from_config() reconstruiria una cabeza distinta
# a la que realmente tiene los pesos guardados, y load_state_dict() fallaria.
modelo_propio.num_classes = NUM_CLASSES_PATH
modelo_propio.usar_global_avg_pool = True
modelo_propio.dropout = None

optimizer_transfer = optim.Adam(modelo_propio.head.parameters(), lr=1e-3)
criterion_transfer = nn.CrossEntropyLoss()

historial_transfer, tiempo_inicio_transfer = entrenar_modelo(
    modelo_propio, train_loader_path, val_loader_path, optimizer_transfer, criterion_transfer,
    device, max_epochs=MAX_EPOCHS, nombre="transfer", modo_backbone_congelado=True,
)


[transfer] epoca  1/25  loss_train=1.6439  f1_val=0.5161
[transfer] epoca  2/25  loss_train=1.2874  f1_val=0.5792
[transfer] epoca  3/25  loss_train=1.1882  f1_val=0.5986
[transfer] epoca  4/25  loss_train=1.1341  f1_val=0.6186
[transfer] epoca  5/25  loss_train=1.0976  f1_val=0.6579
[transfer] epoca  6/25  loss_train=1.0707  f1_val=0.6668
[transfer] epoca  7/25  loss_train=1.0467  f1_val=0.6477
[transfer] epoca  8/25  loss_train=1.0299  f1_val=0.6673
[transfer] epoca  9/25  loss_train=1.0112  f1_val=0.6792
[transfer] epoca 10/25  loss_train=0.9990  f1_val=0.6780
[transfer] epoca 11/25  loss_train=0.9872  f1_val=0.6789
[transfer] epoca 12/25  loss_train=0.9776  f1_val=0.6853
[transfer] epoca 13/25  loss_train=0.9654  f1_val=0.6874
[transfer] epoca 14/25  loss_train=0.9566  f1_val=0.6978
[transfer] epoca 15/25  loss_train=0.9495  f1_val=0.7055
[transfer] epoca 16/25  loss_train=0.9400  f1_val=0.6977
[transfer] epoca 17/25  loss_train=0.9359  f1_val=0.7102
[transfer] epoca 18/25  loss_tr

In [12]:
f1_transfer, y_true_t, y_pred_t = evaluar_f1(modelo_propio, test_loader_path, device)
print(f"F1 macro final (transfer learning): {f1_transfer:.4f}")
print("Cumple el umbral (>= 0.90):", f1_transfer >= 0.75)

checkpoint_transfer = guardar_checkpoint_firmado(
    modelo_propio, f"./entregas/grupo{CODIGO_GRUPO}_transfer.pth",
    historial_transfer, epocas_entrenadas=MAX_EPOCHS, codigo_grupo=CODIGO_GRUPO,
    tiempo_inicio=tiempo_inicio_transfer, batch_fijo=BATCH_FIJO_PATH, device=device,
    componente="transfer",
)


F1 macro final (transfer learning): 0.6413
Cumple el umbral (>= 0.90): False
Checkpoint guardado en ./entregas/grupo0_transfer.pth
  epocas_entrenadas=25  duracion=7096.6s  fingerprint=5d17ac8f7b7b3d31...


## 8. Fine-tuning

A diferencia de ResNet18 (con muchas capas y un `layer4` claramente
identificable), su Chimera es una arquitectura pequeña y arbitraria, no
siempre es directo aislar "el último bloque" en un `nn.Sequential` armado
dinámicamente. Por eso aquí descongelamos **toda la red** para fine-tuning,
pero con un learning rate bajo para no destruir lo que ya aprendió.


In [ ]:
for param in modelo_propio.parameters():
    param.requires_grad = True

optimizer_finetune = optim.Adam([
    {"params": modelo_propio.stem.parameters(), "lr": 1e-5},
    {"params": modelo_propio.features.parameters(), "lr": 1e-5},
    {"params": modelo_propio.head.parameters(), "lr": 1e-4},
])
criterion_finetune = nn.CrossEntropyLoss()

historial_finetune, tiempo_inicio_finetune = entrenar_modelo(
    modelo_propio, train_loader_path, val_loader_path, optimizer_finetune, criterion_finetune,
    device, max_epochs=10, nombre="finetune", modo_backbone_congelado=False,
)


In [ ]:
f1_finetune, y_true_ft, y_pred_ft = evaluar_f1(modelo_propio, test_loader_path, device)
print(f"F1 macro final (fine-tuning): {f1_finetune:.4f}")
print("Cumple el umbral (>= 0.87):", f1_finetune >= 0.76)

checkpoint_finetune = guardar_checkpoint_firmado(
    modelo_propio, f"./entregas/grupo{CODIGO_GRUPO}_finetune.pth",
    historial_finetune, epocas_entrenadas=MAX_EPOCHS, codigo_grupo=CODIGO_GRUPO,
    tiempo_inicio=tiempo_inicio_finetune, batch_fijo=BATCH_FIJO_PATH, device=device,
    componente="finetune",
)


## 9. Comparación final y comprobante de entrega

In [ ]:
print(f"{'Componente':<20}{'F1 macro':>10}{'Cumple umbral':>16}")
print(f"{'Arquitectura propia':<20}{f1_propio:>10.4f}{'N/A (grad.)':>16}")
print(f"{'Transfer learning':<20}{f1_transfer:>10.4f}{str(f1_transfer >= 0.87):>16}")
print(f"{'Fine-tuning':<20}{f1_finetune:>10.4f}{str(f1_finetune >= 0.87):>16}")
print()
print("Presupuesto usado:", total_pts, f"/ rango {PRESUPUESTO_MIN}-{PRESUPUESTO_MAX}")
print("Secuencia de bloques:", MI_BLOQUES)


### Cómo entregan

1. Suban los tres archivos `.pth` que se generaron en `./entregas/` (uno por componente) al espacio indicado (carpeta compartida /Moodle) **no por correo ni copiados a mano**: los archivos ya traen la    configuración de arquitectura, el número de épocas, los timestamps de    inicio/fin y un hash de verificación (`fingerprint_sha256`) calculado sobre las predicciones de su modelo en un batch fijo. Ese hash cambia si los pesos cambian por cualquier motivo, ojo no es editable a mano.
2. 


In [ ]:
comprobante = {
    "codigo_grupo": CODIGO_GRUPO,
    "configuracion": {
        "bloques": MI_BLOQUES, "optimizador": MI_OPTIMIZADOR, "scheduler": MI_SCHEDULER,
        "loss": MI_LOSS, "conv_custom": MI_CONV_CUSTOM, "pooling_extra": MI_POOLING_EXTRA,
        "global_avg_pool": MI_GLOBAL_AVG_POOL, "dropout": MI_DROPOUT,
    },
    "presupuesto_total": total_pts,
    "resultados": {
        "f1_arquitectura_propia": round(f1_propio, 4),
        "f1_transfer": round(f1_transfer, 4),
        "f1_finetune": round(f1_finetune, 4),
    },
    "fingerprints": {
        "arquitectura_propia": checkpoint_propio["fingerprint_sha256"],
        "transfer": checkpoint_transfer["fingerprint_sha256"],
        "finetune": checkpoint_finetune["fingerprint_sha256"],
    },
}

with open(f"./entregas/grupo{CODIGO_GRUPO}_comprobante.json", "w", encoding="utf-8") as f:
    json.dump(comprobante, f, indent=2, ensure_ascii=False)

print("Comprobante guardado en", f"./entregas/grupo{CODIGO_GRUPO}_comprobante.json")
print(json.dumps(comprobante, indent=2, ensure_ascii=False))


### Checklist final

- [ ] `CODIGO_GRUPO` fue reemplazado por el código real del grupo.
- [ ] La celda de `calcular_presupuesto` de su arquitectura corrió sin
  `AssertionError` (150-200 pts).
- [ ] Los tres componentes se entrenaron con `MAX_EPOCHS=25` (no editado).
- [ ] Existen los tres `.pth` en `./entregas/` y el `comprobante.json`.
- [ ] El notebook corre de arriba a abajo sin editar celdas fuera de las
  marcadas con `TODO (grupo)`.

### Nota sobre registro automático (opcional, para el profesor)

Si quieres centralizar los comprobantes sin exponer ninguna credencial en el
notebook: crea un Google Form con un campo de texto por cada clave del
`comprobante` (o un solo campo de texto largo para pegar el JSON completo).
Los Forms de Google aceptan envíos anónimos a su endpoint público
`.../formResponse` sin necesidad de API key ni OAuth — no hay ningún secreto
que un estudiante pueda leer en el código. Se puede automatizar con:

```python
import requests
requests.post(
    "https://docs.google.com/forms/d/e/TU_FORM_ID/formResponse",
    data={"entry.123456789": json.dumps(comprobante)},  # entry.XXXX: se obtiene
                                                          # inspeccionando el HTML
                                                          # de tu propio formulario
)
```

Esto no se incluyó activo en el notebook porque depende de un Form que solo
tú puedes crear (necesita tu Google Drive). Si lo armas, el mismo bloque de
código de arriba se puede pegar tal cual al final del notebook.
